# ViT-Small -- static pruning (CIFAR-10 @ 224px)

**The protocol is not restated here on purpose.** Cell 3 below prints
`nb.FAMILIES['vit-small']` for all three phases, and that is the only source of
truth. A table in this cell can only drift out of date -- and did: it claimed
60 epochs, `delta_T=176` and a 50-epoch fine-tune, where the code runs 50, 175
and 25. Note `45000 // 256 == 175` exactly, so 176 would have pushed every
prune event off the epoch boundary. Read the printed config, not prose.

Inputs are 224x224 -- 49x the pixels of the CNN notebooks -- so every phase
here costs far more than the CIFAR-scale runs. Weights come pretrained from the
HF hub (`MODELS['vit-small'].weight` is empty by design, and preflight fetches
them); DyReLU is unavailable for HF models by construction. Sources:
`docs/foundation.md`.

## Read before running

**1. You are this path's first real user.** A full dense -> prune(0.95) ->
BaCP(0.95) chain has been executed end to end on CPU and wrote `ok` records, so
the code works. But no ViT cell has ever had a real training run.

**2. Do not seed this from the smoke notebook.** `resolve_checkpoint`
(`project/results.py`) matches on status, model, dataset, method and seed but
does **not** filter smoke records. A sparse cell run after `00_smoke_test_all`
can silently initialise from a checkpoint trained on 2 batches for 2 epochs,
and record the result under a real key with no warning. Run the dense cell in
*this* notebook first and let it finish.

**3. ViT sparsity is not comparable to the CNN rows yet.** Patch-embedding,
position embeddings and `cls_token` are excluded from the prunable set by
`_EXCLUDE_EMBEDDINGS` (`project/pruning_factory.py:35`), whose stated rationale
concerns LLM tied embeddings but which applies unconditionally. `vit-tiny` is
96.09% prunable against `vgg19`'s 99.99%, so a nominal 0.999 cell reaches a
true sparsity near 0.960. Do not place these numbers beside the CNN rows until
that convention is settled.


In [ ]:
import sys, pathlib

# Find nb_common.py whether the kernel started in this folder or at the repo root.
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')

import nb_common as nb
info = nb.setup()

## Configure

`SMOKE=True` runs every cell below on 2 batches first -- do that once on a new cluster before real training.

In [ ]:
MODEL      = 'vit-small'
SEED       = 1
GPU        = 0
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
SMOKE      = False
OVERRIDES  = {}       # e.g. dict(epochs=3) to shorten every run below

for phase in ('dense', 'prune', 'bacp'):
    print(f'{phase:>6}: ', {**nb.FAMILIES[MODEL]['base'], **nb.FAMILIES[MODEL][phase]})

## Weights + preflight

Halts before any GPU time is spent if the model is not actually pretrained (`load_weights` fails soft, so a missing checkpoint would otherwise silently train from random init).

In [ ]:
nb.preflight(MODEL, num_classes=nb.FAMILIES[MODEL]['base']['num_classes'])

## Dense baseline (required first)

Every sparse run below starts from this checkpoint (same seed). Re-running skips it if its record exists; delete the record under `results/runs/` to re-arm.

In [ ]:
dense = nb.make_cell(MODEL, 'dense', seed=SEED, smoke=SMOKE, **OVERRIDES)
out = nb.run(dense, gpu=GPU)

## I.P. — magnitude (Han et al. 2015)

Iterative pruning + recovery, the sparse baseline. One run per sparsity level, streamed back to back.

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='magnitude', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## I.P. — SNIP-it (Lee et al. 2019 / Verdenius et al. 2020)

Iterative pruning + recovery, the sparse baseline. One run per sparsity level, streamed back to back.

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='snip', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## I.P. — WANDA (Sun et al. 2023)

Iterative pruning + recovery, the sparse baseline. One run per sparsity level, streamed back to back.

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='wanda', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP — magnitude

The contrastive objective (PrC/SnC/FiC + CE, lambdas 0.25 each, tau 0.15 -- `docs/foundation.md` SS2-3), then AdamW finetune.

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='magnitude', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP — SNIP-it

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='snip', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP — WANDA

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='wanda', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## Results vs the paper

`-` = not run yet (ours) or not published (paper).

In [ ]:
nb.results_table(MODEL)

## Health

Every static record for this model. Delete a record to re-arm its cell.

In [ ]:
import runner as R
done = sorted(k for k in R.completed_keys() if k.startswith('static.') and MODEL in k)
print(f'{len(done)} static record(s) for {MODEL}:')
for k in done:
    print(' ', k)